In [58]:
import pandas as pd
import numpy as np
import plotly.express as px

In [18]:
loan_original = pd.read_csv("data/Loans.csv")

C:\Users\Shaoxuan Zhang\AppData\Local\Temp\ipykernel_16776\502106635.py:1: DtypeWarning: Columns (2: desc) have mixed types. Specify dtype option on import or set low_memory=False.
  loan_original = pd.read_csv("data/Loans.csv")


In [19]:
loan_original.head(5)

,id,customer_id,desc,dti,dti_joint,installment,int_rate,issue_d,loan_amnt,loan_status,out_prncp,purpose,total_rec_int,total_rec_prncp
0,f9b93f05-0175-4f70-85d5-cfafc232161e,03d84961-f689-416a-af0a-3b0d3ea8a31d,NaN,18.24,NaN,84.92,13.56,2013,2500,Current,2386.02,debt_consolidation,53.04,113.98
1,50d7cae8-72ed-49d7-b09a-0313462a973f,7e28a4cb-cc30-4520-86f1-becd0cb46d12,NaN,26.52,NaN,777.23,18.94,2020,30000,Current,29387.75,debt_consolidation,894.86,612.25
2,2b61136e-27af-42ae-a818-9f1f92e8b847,55aba73f-5906-467e-9c8a-7e69cb649704,NaN,10.51,NaN,180.69,17.97,2019,5000,Current,4787.21,debt_consolidation,141.10,212.79
3,1aac1df0-1456-4014-ba5e-250c5767a392,6b09bad3-d579-4ac9-ac0c-5a557a7a19bc,NaN,16.74,NaN,146.51,18.94,2021,4000,Current,3831.93,debt_consolidation,118.64,168.07
4,70e77fff-0c29-4167-84bf-03f9affafc66,03cb5a91-f2c6-4e25-a837-f02ccac62e39,NaN,26.35,NaN,731.78,16.14,2013,30000,Current,29339.02,debt_consolidation,762.23,660.98


- dti: debt-to-income retio - Borrower's debt relative to income (VH, credit risk)
- installment - Amount the customer was originally required to pay monthly (H, repayment analysis)
- int_rate - interest rate charged on the loan (VH, profitability and risk)
- issue_d : year origination (trends)
- loan_amnt: Original amount lent to the customer (VH, original exposure)
- out-prncp: Amount of original principal still unpaid (current exposure)
- total_rec_int: interest received to date (Interest payments collected so far)
- total_rec_prncp: Principal repayments collected so far


### 1. Loan portfolio data cleansing & DQ analysis

In [40]:
print("Dataset overview")
print(f"Number of records: {len(loan_original):,}")
print(f"Number of columns: {len(loan_original.columns)}")

print("\nTotal loan amount:")
print(loan_original["loan_amnt"].sum())


Dataset overview
Number of records: 2,260,668
Number of columns: 14

Total loan amount:
34016115925


In [21]:
print("\nLoan status distribution:")
print(loan_original["loan_status"].value_counts())


Loan status distribution:
loan_status
Fully Paid                                             1041952
Current                                                 919695
Charged Off                                             261655
Late (31-120 days)                                       21897
In Grace Period                                           8952
Late (16-30 days)                                         3737
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     31
Name: count, dtype: int64


In [22]:
# Missing values
missing = pd.DataFrame({
    "Missing values": loan_original.isnull().sum(),
    "Missing (%)": loan_original.isnull().mean() * 100
})

missing.sort_values("Missing (%)", ascending=False)

,Missing values,Missing (%)
dti_joint,2139962,94.660605
desc,2134603,94.423551
dti,1711,0.075686
id,0,0.000000
customer_id,0,0.000000
installment,0,0.000000
int_rate,0,0.000000
issue_d,0,0.000000
loan_amnt,0,0.000000
loan_status,0,0.000000


In [44]:
print("Duplicate customers:", loan_original["customer_id"].duplicated().sum())

print("Duplicate rows:", loan_original.duplicated().sum())

Duplicate customers: 1
Duplicate rows: 0


In [46]:
# check negative value
numeric_cols = loan_original.select_dtypes(include="number").columns

(loan_original[numeric_cols] <= 0).sum()

dti                   1734
dti_joint               18
installment              0
int_rate                 0
issue_d                  0
loan_amnt                0
out_prncp          1312200
total_rec_int         2657
total_rec_prncp       2575
dtype: int64

In [50]:
# DTI quality by loan status
loan_original.groupby("loan_status").agg(
    Total_loans=("dti", "size"),
    Missing_DTI=("dti", lambda x: x.isna().sum()),
    Zero_DTI=("dti", lambda x: (x == 0).sum()),
    Negative_DTI=("dti", lambda x: (x < 0).sum())
)

,Total_loans,Missing_DTI,Zero_DTI,Negative_DTI
loan_status,,,,
Charged Off,261655,62,158,0
Current,919695,1357,848,0
Default,31,0,0,0
Does not meet the credit policy. Status:Charged Off,761,0,11,0
Does not meet the credit policy. Status:Fully Paid,1988,0,11,0
Fully Paid,1041952,250,657,2
In Grace Period,8952,6,11,0
Late (16-30 days),3737,8,7,0
Late (31-120 days),21897,28,29,0


In [49]:
zero_balance = loan_original[
    loan_original["out_prncp"] == 0
]

zero_balance["loan_status"].value_counts()

loan_status
Fully Paid                                             1041952
Charged Off                                             261655
Current                                                   5655
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
In Grace Period                                            111
Late (31-120 days)                                          51
Late (16-30 days)                                           27
Name: count, dtype: int64

Summary of findings so far:
- More than 1000 missing values in dti, 2 negative dti values, 1700 records of dti=0
- Majority of dti issues in current, fully paid and charged off loans
- large amount of missing values in dti_joint but not necessarily a problem if most of the loans were individual applications
- over 1.3 million zero outstanding principle while 1.04 million fully paid loans


### 2. Client profile analysis

Business question: Does Bank B's sample adequately represent the geographical distribution of its loan portfolio?

In [52]:
customer_info = pd.read_csv("data/Customers.csv")

In [53]:
print("Unique loan customers:", loan_original["customer_id"].nunique())
print("Unique customer records:", customer_info["customer_id"].nunique())

# Check duplicate customer IDs in customer information
print("Duplicate customer IDs:",
      customer_info["customer_id"].duplicated().sum())

Unique loan customers: 2260667
Unique customer records: 2258380
Duplicate customer IDs: 1


In [59]:
duplicates = customer_info[
    customer_info["customer_id"].duplicated(keep=False)
].sort_values("customer_id")

duplicates

,customer_id,emp_title,emp_length,annual_inc,annual_inc_joint,home_ownership,total_acc,zip_code,addr_state,verification_status
108729,771451e2-af1d-4747-bc78-54f5e396d1a1,assistant manager,3 years,69762.0,NaN,RENT,13.0,330xx,FL,Not Verified
1567713,771451e2-af1d-4747-bc78-54f5e396d1a1,Sewing,3 years,35000.0,NaN,RENT,15.0,906xx,CA,Source Verified


In [65]:
loan_analysis = loan_original.merge(
    customer_info[["customer_id", "emp_title", "annual_inc", "addr_state"]]
        .drop_duplicates(subset="customer_id"),
    on="customer_id",
    how="left",
    indicator=True,
    validate="many_to_one"
)

In [61]:
print("Original loan records:", len(loan_original))
print("Merged loan records:", len(loan_analysis))

print(loan_analysis["_merge"].value_counts())

Original loan records: 2260668
Merged loan records: 2260668
_merge
both          2258381
left_only        2287
right_only          0
Name: count, dtype: int64


In [62]:
unmatched_loans = loan_analysis[
    loan_analysis["_merge"] == "left_only"
]

print("Unmatched loan records:", len(unmatched_loans))
print("Unique unmatched customers:",
      unmatched_loans["customer_id"].nunique())

Unmatched loan records: 2287
Unique unmatched customers: 2287


In [64]:
unmatched_loans["loan_status"].value_counts()

loan_status
Fully Paid                                             1227
Current                                                 780
Charged Off                                             235
Late (31-120 days)                                       20
Does not meet the credit policy. Status:Fully Paid       15
In Grace Period                                           5
Late (16-30 days)                                         3
Does not meet the credit policy. Status:Charged Off       2
Name: count, dtype: int64

2287 loan records have no matching customer information, 1 customer has conflicting personal information
Due diligence implcation:
- Customer-level: identification and data integrity
- Unmatched records: Incomplete customer-level information (0.1%) for a small proportion of the portfolio
- unmatched records distributed amongst fully paid, current and charged off, potential credit losses

In [70]:
state_summary = loan_analysis.groupby("addr_state").agg(
    Number_of_loans=("id", "count"),
    Unique_customers=("customer_id", "nunique"),
    Total_outstanding=(
        "out_prncp", "sum"
    )
).sort_values("Number_of_loans", ascending=False)

state_summary["Loan_percentage"] = (
    state_summary["Number_of_loans"] /
    state_summary["Number_of_loans"].sum() * 100
).round(2)

In [71]:
state_summary.head(10)
print(len(state_summary))

43


In [76]:
fig = px.choropleth(
    state_summary.reset_index(),
    locations="addr_state",
    locationmode="USA-states",
    color="Number_of_loans",
    scope="usa",
    color_continuous_scale="Blues",
    hover_name="addr_state",
    hover_data=[
        "Number_of_loans",
        "Unique_customers",
        "Total_outstanding",
        "Loan_percentage"
    ],
    title="Geographical Distribution of Loans Across US States"
)

fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

### 3. Category

In [48]:
# Outstanding principal by loan status
loan_original.groupby("loan_status")["out_prncp"].agg(
    ["count", "sum", "mean"]
).sort_values("sum", ascending=False)

,count,sum,mean
loan_status,,,
Current,919695,9.665843e+09,10509.835669
Late (31-120 days),21897,2.456080e+08,11216.512682
In Grace Period,8952,9.868228e+07,11023.489641
Late (16-30 days),3737,4.110787e+07,11000.233257
Default,31,3.505956e+05,11309.536452
Charged Off,261655,0.000000e+00,0.000000
Does not meet the credit policy. Status:Charged Off,761,0.000000e+00,0.000000
Does not meet the credit policy. Status:Fully Paid,1988,0.000000e+00,0.000000
Fully Paid,1041952,0.000000e+00,0.000000


In [51]:
# Fully paid loan
fully_paid = loan_original[loan_original["loan_status"] == "Fully Paid"]

In [36]:
numeric_cols = fully_paid.select_dtypes(include="number").columns

(fully_paid[numeric_cols] == 0).sum()

dti                    657
dti_joint                6
installment              0
int_rate                 0
issue_d                  0
loan_amnt                0
out_prncp          1041952
total_rec_int          183
total_rec_prncp          0
dtype: int64